## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score, precision_score, recall_score,
    brier_score_loss, average_precision_score, matthews_corrcoef
)
from sklearn.base import BaseEstimator, TransformerMixin

sns.set_theme(style="whitegrid")

## 2. Load Data and Apply the Same Train/Test Split as main.ipynb

To ensure fair comparison with all other models, we replicate the **exact** preprocessing
and `train_test_split` from `main.ipynb` (80/20, `random_state=42`, stratified).
We then expand the continuous predictors into 2nd-degree polynomial features
to give PCR/PLS plenty of predictors to work with, as required.

In [ ]:
# --- Load & clean (identical to main.ipynb) ---
full_dataset = pd.read_csv("dataset.csv").dropna()
dataset = full_dataset[full_dataset["Profession"] == "Student"].reset_index(drop=True)
dataset = dataset.drop(columns=["id", "City", "Profession", "Work Pressure", "Job Satisfaction"])
dataset = dataset[dataset["Sleep Duration"] != "Others"].copy()
dataset = dataset[dataset["Dietary Habits"] != "Others"].copy()

# --- Separate target, drop suicidal ideation (same as primary models in main.ipynb) ---
y = dataset["Depression"]
X = dataset.drop(columns=["Depression", "Have you ever had suicidal thoughts ?"])

# --- Train/test split: 80/20, stratified, random_state=42 — matches main.ipynb exactly ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} observations")
print(f"Test set:     {X_test.shape[0]} observations")
print(f"Depression prevalence — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

## 3. Feature Expansion — "Plenty of Predictors"

We use `PolynomialFeatures(degree=2)` on the continuous predictors to generate squared terms
and interaction terms (e.g., Age × CGPA). Combined with one-hot encoded categoricals this
brings us to 60+ predictors — giving PCR and PLS a genuinely high-dimensional problem to solve.

**Important:** The preprocessor is fit **only on the training set** and then applied to the
test set, preventing any data leakage from test observations into the feature scaling or PCA/PLS decomposition.

In [ ]:
continuous_features = ["Age", "CGPA", "Academic Pressure", "Work/Study Hours", "Financial Stress", "Study Satisfaction"]
categorical_features = ["Gender", "Sleep Duration", "Dietary Habits", "Degree", "Family History of Mental Illness"]

# Polynomial expansion on continuous features only, then re-scale
preprocessor = ColumnTransformer(
    transformers=[
        ("num_poly", Pipeline([
            ("scaler1", StandardScaler()),
            ("poly",    PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler2", StandardScaler())
        ]), continuous_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

# Fit on training data only, then transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)   # <-- no leakage: uses train-fit params

print(f"Original continuous features : {len(continuous_features)}")
print(f"Total predictors after expansion: {X_train_processed.shape[1]}")
print(f"Processed training shape : {X_train_processed.shape}")
print(f"Processed test shape     : {X_test_processed.shape}")

## 4. Shared Evaluation Helper

A single function computes all metrics used in the main notebook's comparison table
(ROC-AUC, Brier score, accuracy, precision, recall, F1, MCC, AP), so the numbers
slot directly into that table.

In [ ]:
def evaluate_on_test(name, y_true, y_pred, y_proba):
    """Print and return the standard metric set used across all models."""
    metrics = {
        "Model"       : name,
        "ROC-AUC"     : round(roc_auc_score(y_true, y_proba), 4),
        "Brier_Score" : round(brier_score_loss(y_true, y_proba), 4),
        "Accuracy"    : round(accuracy_score(y_true, y_pred), 4),
        "Precision"   : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"      : round(recall_score(y_true, y_pred), 4),
        "F1_Score"    : round(f1_score(y_true, y_pred), 4),
        "MCC"         : round(matthews_corrcoef(y_true, y_pred), 4),
        "AP"          : round(average_precision_score(y_true, y_proba), 4),
    }
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in metrics.items():
        if k != "Model":
            print(f"  {k:<14}: {v}")
    return metrics

## 5. Principal Component Regression (PCR)

PCR is a two-step pipeline: PCA compresses the 60+ predictors into the smallest set of
components that retains 95% of variance, then logistic regression classifies on those components.
Hyperparameter tuning uses 5-fold CV **on the training set only**; final metrics are reported
on the held-out test set.

In [ ]:
# Build PCR pipeline (preprocessor already fit; work directly on processed arrays)
pcr_pipeline = Pipeline([
    ("pca",        PCA(n_components=0.95, random_state=42)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))
])

# 5-fold CV on training set to report generalisation AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pcr_cv_auc = cross_val_score(pcr_pipeline, X_train_processed, y_train, cv=cv, scoring="roc_auc")
print(f"PCR — Mean 5-Fold CV ROC-AUC (train): {pcr_cv_auc.mean():.4f} (+/- {pcr_cv_auc.std():.4f})")

# Fit on full training set, evaluate on held-out test set
pcr_pipeline.fit(X_train_processed, y_train)
pcr_pred  = pcr_pipeline.predict(X_test_processed)
pcr_proba = pcr_pipeline.predict_proba(X_test_processed)[:, 1]

n_components_kept = pcr_pipeline.named_steps["pca"].n_components_
print(f"PCA reduced {X_train_processed.shape[1]} predictors → {n_components_kept} components (95% variance)")

pcr_metrics = evaluate_on_test("PCR (No Suicide)", y_test, pcr_pred, pcr_proba)

## 6. Partial Least Squares (PLS-DA)

Unlike PCA (which compresses blindly by variance), PLS finds components that maximally
correlate with the target variable. We wrap `PLSRegression` in a custom transformer so it
fits cleanly inside a sklearn `Pipeline`, then feed the compressed components into logistic
regression. The number of components is a tunable hyperparameter; 10 was found to perform
well in preliminary CV testing.

In [ ]:
class PLSTransformer(BaseEstimator, TransformerMixin):
    """Wraps PLSRegression so it behaves as a sklearn transformer."""
    def __init__(self, n_components=10):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)


pls_pipeline = Pipeline([
    ("pls",        PLSTransformer(n_components=10)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))
])

# 5-fold CV on training set
pls_cv_auc = cross_val_score(pls_pipeline, X_train_processed, y_train, cv=cv, scoring="roc_auc")
print(f"PLS — Mean 5-Fold CV ROC-AUC (train): {pls_cv_auc.mean():.4f} (+/- {pls_cv_auc.std():.4f})")

# Fit on full training set, evaluate on held-out test set
pls_pipeline.fit(X_train_processed, y_train)
pls_pred  = pls_pipeline.predict(X_test_processed)
pls_proba = pls_pipeline.predict_proba(X_test_processed)[:, 1]

pls_metrics = evaluate_on_test("PLS-DA (No Suicide)", y_test, pls_pred, pls_proba)

## 7. Visual Comparison — Confusion Matrices & Metrics Table

Both confusion matrices are now computed from predictions on the **same held-out test set**
used by every other model in the project, making the counts directly comparable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, title, cmap in zip(
    axes,
    [pcr_pred, pls_pred],
    ["PCR Confusion Matrix", "PLS Confusion Matrix"],
    ["Blues", "Greens"]
):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Not Depressed", "Depressed"]).plot(
        cmap=cmap, ax=ax, colorbar=False
    )
    ax.set_title(title)
    ax.grid(False)

plt.suptitle("PCR vs PLS — Held-Out Test Set (20%)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side metrics table for easy copy-paste into the report
results_df = pd.DataFrame([pcr_metrics, pls_metrics]).set_index("Model")
print("\nModel Comparison — Test Set Performance")
print(results_df.to_string())